In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/processed/9558")

lap_data = pd.read_parquet(
    DATA_DIR / "lap_performance.parquet"
)

fuel_features = pd.read_parquet(
    DATA_DIR / "fuel_features.parquet"
)

traffic_features = pd.read_parquet(
    DATA_DIR / "traffic_features.parquet"
)

tyre_features = pd.read_parquet(
    DATA_DIR / "tyre_features.parquet"
)

print("Lap data:", lap_data.shape)
print("Fuel features:", fuel_features.shape)
print("Traffic features:", traffic_features.shape)
print("Tyre features:", tyre_features.shape)

Lap data: (962, 19)
Fuel features: (692, 3)
Traffic features: (962, 6)
Tyre features: (650, 4)


In [3]:
datasets = {
    "lap_performance": lap_data,
    "fuel_features": fuel_features,
    "traffic_features": traffic_features,
    "tyre_features": tyre_features,
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(df.columns.tolist())

print("\n===== LAP PERFORMANCE SAMPLE =====")
print(lap_data.head(2))

print("\n===== FUEL FEATURES SAMPLE =====")
print(fuel_features.head(2))

print("\n===== TRAFFIC FEATURES SAMPLE =====")
print(traffic_features.head(2))

print("\n===== TYRE FEATURES SAMPLE =====")
print(tyre_features.head(2))


lap_performance
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age']

fuel_features
['driver_number', 'lap_number', 'fuel_effect']

traffic_features
['driver_number', 'lap_number', 'position', 'traffic_ahead', 'traffic_behind', 'position_change']

tyre_features
['driver_number', 'lap_number', 'tyre_age', 'tyre_degradation_effect']

===== LAP PERFORMANCE SAMPLE =====
   meeting_key  session_key  driver_number  lap_number  \
0         1240         9558              1           1   
1         1240         9558              1           2   

                        date_start  duration_sector_1  duration_sector_2  \
0 2024-07-07 14:03:12.540000+00:00             33.528             37.975   
1 2024-07-07 14:04:49.750000+00:00          

In [4]:
print("Lap performance duplicate keys:",
      lap_data.duplicated(["driver_number", "lap_number"]).sum())

print("Fuel feature duplicate keys:",
      fuel_features.duplicated(["driver_number", "lap_number"]).sum())

print("Traffic feature duplicate keys:",
      traffic_features.duplicated(["driver_number", "lap_number"]).sum())

print("Tyre feature duplicate keys:",
      tyre_features.duplicated(["driver_number", "lap_number"]).sum())

Lap performance duplicate keys: 0
Fuel feature duplicate keys: 0
Traffic feature duplicate keys: 1
Tyre feature duplicate keys: 0


In [5]:
lap_keys = lap_data[["driver_number", "lap_number"]].drop_duplicates()

fuel_keys = fuel_features[["driver_number", "lap_number"]]
traffic_keys = traffic_features[["driver_number", "lap_number"]]
tyre_keys = tyre_features[["driver_number","lap_number"]]

print("Fuel coverage:",
      lap_keys.merge(
          fuel_keys,
          on=["driver_number", "lap_number"],
          how="inner"
      ).shape[0])

print("Traffic coverage:",
      lap_keys.merge(
          traffic_keys,
          on=["driver_number","lap_number"],
          how="inner"
      ).shape[0])

print("Tyre coverage:",
      lap_keys.merge(
          tyre_keys,
          on=["driver_number", "lap_number"],
          how="inner"
      ).shape[0])

Fuel coverage: 692
Traffic coverage: 962
Tyre coverage: 650


In [6]:
traffic_features = traffic_features.drop_duplicates(
    subset=["driver_number", "lap_number"]
).copy()

print(
    "Traffic duplicate keys:",
    traffic_features.duplicated(
        ["driver_number", "lap_number"]
    ).sum()
)

print("Traffic features shape:", traffic_features.shape)

Traffic duplicate keys: 0
Traffic features shape: (961, 6)


In [7]:
enhanced_lap_data = lap_data.copy()

enhanced_lap_data = enhanced_lap_data.merge(
    fuel_features,
    on=["driver_number","lap_number"],
    how="left",
    validate="one_to_one"
)

enhanced_lap_data = enhanced_lap_data.merge(
    traffic_features,
    on=["driver_number","lap_number"],
    how="left",
    validate="one_to_one"
)

enhanced_lap_data = enhanced_lap_data.merge(
    tyre_features,
    on=["driver_number","lap_number"],
    how="left",
    validate="one_to_one"
)
print("Enhanced pace data shape:", enhanced_lap_data.shape)
print("\nColumns:")
print(enhanced_lap_data.columns.tolist())


Enhanced pace data shape: (962, 26)

Columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age_x', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_age_y', 'tyre_degradation_effect']


In [8]:
print(
    enhanced_lap_data[
        [
            "fuel_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "tyre_degradation_effect"
        ]
    ].isna().sum()
)

fuel_effect                270
traffic_ahead                1
traffic_behind               1
position_change              1
tyre_degradation_effect    312
dtype: int64


In [9]:
tyre_age_check = enhanced_lap_data[
    ["tyre_age_x", "tyre_age_y"]
].dropna()

print("Rows with both tyre ages:", len(tyre_age_check))

print("Mismatched tyre ages:",
      (
          tyre_age_check["tyre_age_x"]
          != tyre_age_check["tyre_age_y"]
      ).sum()
)

Rows with both tyre ages: 650
Mismatched tyre ages: 0


In [10]:
enhanced_lap_data = enhanced_lap_data.drop(
    columns=["tyre_age_y"]
).rename(
    columns={"tyre_age_x": "tyre_age"}
)

print(enhanced_lap_data.columns.tolist())
print("\nShape:", enhanced_lap_data.shape)

['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect']

Shape: (962, 25)


In [11]:
missing_fuel = enhanced_lap_data[
    enhanced_lap_data["fuel_effect"].isna()
]

missing_tyre = enhanced_lap_data[
    enhanced_lap_data["tyre_degradation_effect"].isna()
]

missing_traffic = enhanced_lap_data[
    enhanced_lap_data["traffic_ahead"].isna()
]

print("Missing fuel:", len(missing_fuel))
print("Missing tyre:", len(missing_tyre))
print("Missing traffic:", len(missing_traffic))

Missing fuel: 270
Missing tyre: 312
Missing traffic: 1


In [12]:
print(
    missing_traffic[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "position",
            "compound",
            "tyre_age"
        ]
    ]
)

print(
    missing_fuel[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound",
            "tyre_age"
        ]
    ].head(10)
)

print(
    missing_tyre[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound",
            "tyre_age"
        ]
    ].head(10)
)


     driver_number  lap_number  lap_duration  position compound  tyre_age
208             11           1           NaN       NaN     HARD       0.0
    driver_number  lap_number  lap_duration      compound  tyre_age
26              1          27       119.169  INTERMEDIATE       0.0
27              1          28       101.176  INTERMEDIATE       1.0
28              1          29       101.533  INTERMEDIATE       2.0
29              1          30       102.098  INTERMEDIATE       3.0
30              1          31       102.025  INTERMEDIATE       4.0
31              1          32       102.538  INTERMEDIATE       5.0
32              1          33       102.606  INTERMEDIATE       6.0
33              1          34       102.447  INTERMEDIATE       7.0
34              1          35       103.029  INTERMEDIATE       8.0
35              1          36       101.958  INTERMEDIATE       9.0
    driver_number  lap_number  lap_duration      compound  tyre_age
0               1           1       

In [13]:
print("===== COMPOUND COUNTS =====")
print(
    enhanced_lap_data["compound"]
    .value_counts(dropna=False)
)

print("\n===== PIT-OUT LAPS =====")
print(
    enhanced_lap_data["is_pit_out_lap"]
    .value_counts(dropna=False)
)

print("\n===== MISSING LAP DURATION =====")
print(
    enhanced_lap_data["lap_duration"].isna().sum()
)

print("\n===== LAP DURATION SUMMARY =====")
print(
    enhanced_lap_data["lap_duration"].describe()
)

===== COMPOUND COUNTS =====
compound
MEDIUM          508
INTERMEDIATE    234
SOFT            174
HARD             44
NaN               2
Name: count, dtype: int64

===== PIT-OUT LAPS =====
is_pit_out_lap
False    917
True      45
Name: count, dtype: int64

===== MISSING LAP DURATION =====
3

===== LAP DURATION SUMMARY =====
count    959.000000
mean      96.833269
std        7.612357
min       88.293000
25%       91.717000
50%       93.219000
75%      101.995000
max      128.459000
Name: lap_duration, dtype: float64


In [15]:
print("===== PIT-OUT COLUMN INFO =====")
print("dtype:", enhanced_lap_data["is_pit_out_lap"].dtype)

print("\nUnique values:")
print(enhanced_lap_data["is_pit_out_lap"].unique())

print("\nValue counts:")
print(enhanced_lap_data["is_pit_out_lap"].value_counts(dropna=False))

===== PIT-OUT COLUMN INFO =====
dtype: bool

Unique values:
[False  True]

Value counts:
is_pit_out_lap
False    917
True      45
Name: count, dtype: int64


In [16]:
print("Total rows:", len(enhanced_lap_data))

step1 = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
].copy()

step2 = step1[
    step1["is_pit_out_lap"] == False
].copy()

print("After removing pit out:", len(step2))

step3 = step2[
    step2["compound"].isin(["SOFT","MEDIUM","HARD"])
].copy()

print("After dry compounds:", len(step3))

step4 = step3[
    ~step3["lap_number"].isin([19,25])
].copy()

print("After removing low-grip laps:", len(step4))

print("\nFinal pit-out counts:")
print(step4["is_pit_out_lap"].value_counts())

print("\nFinal compound counts:")
print(step4["compound"].value_counts())


Total rows: 962
After removing pit out: 914
After dry compounds: 703
After removing low-grip laps: 668

Final pit-out counts:
is_pit_out_lap
False    668
Name: count, dtype: int64

Final compound counts:
compound
MEDIUM    467
SOFT      161
HARD       40
Name: count, dtype: int64


In [17]:
pace_model_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & enhanced_lap_data["compound"].isin(["SOFT", "MEDIUM", "HARD"])
    & (~enhanced_lap_data["lap_number"].isin([19, 25]))
].copy()

print("===== PACE MODEL DATA =====")
print("Shape:", pace_model_data.shape)

print("\n===== COMPOUNDS =====")
print(pace_model_data["compound"].value_counts())

print("\n===== PIT-OUT LAPS =====")
print(pace_model_data["is_pit_out_lap"].value_counts())

print("\n===== MISSING LAP DURATION =====")
print(pace_model_data["lap_duration"].isna().sum())

===== PACE MODEL DATA =====
Shape: (668, 25)

===== COMPOUNDS =====
compound
MEDIUM    467
SOFT      161
HARD       40
Name: count, dtype: int64

===== PIT-OUT LAPS =====
is_pit_out_lap
False    668
Name: count, dtype: int64

===== MISSING LAP DURATION =====
0


In [20]:
feature_columns = [
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect"
]

for column in feature_columns:
    missing = pace_model_data[column].isna().sum()
    available = pace_model_data[column].notna().sum()
    total = len(pace_model_data)

    print(
        f"{column}: "
        f"{available}/{total} available "
        f"({available / total * 100:.1f}%), "
        f"{missing} missing"
    )

print(
    pace_model_data[
        pace_model_data["fuel_effect"].isna()
    ][
        ["driver_number", "lap_number", "compound", "tyre_age", "lap_duration"]
    ].to_string(index=False)
)

print(
    pace_model_data[
        pace_model_data["tyre_degradation_effect"].isna()
    ][
        ["driver_number", "lap_number", "compound", "tyre_age", "lap_duration"]
    ].to_string(index=False)
)

fuel_effect: 657/668 available (98.4%), 11 missing
traffic_ahead: 668/668 available (100.0%), 0 missing
traffic_behind: 668/668 available (100.0%), 0 missing
position_change: 668/668 available (100.0%), 0 missing
tyre_degradation_effect: 650/668 available (97.3%), 18 missing
 driver_number  lap_number compound  tyre_age  lap_duration
             2          27   MEDIUM      26.0       112.246
             4          27   MEDIUM      26.0       104.806
            14          27   MEDIUM      26.0       108.513
            20          27   MEDIUM      26.0       111.325
            22          27   MEDIUM      26.0       108.126
            23          27   MEDIUM      26.0       107.754
            31          26   MEDIUM       4.0       118.710
            44          27   MEDIUM      26.0       105.183
            63          27   MEDIUM      26.0       107.698
            81          27   MEDIUM      26.0       109.747
            81          28   MEDIUM      27.0       112.976
 dri

In [22]:
fuel_coefficient = -0.084122

missing_fuel = pace_model_data["fuel_effect"].isna()

pace_model_data.loc[missing_fuel,"fuel_effect"] = (
    (pace_model_data.loc[missing_fuel, "lap_number"] - 1)*fuel_coefficient
)

print("Missing fuel_effect after filling:",
    pace_model_data["fuel_effect"].isna().sum())

print("Filled fuel effects:")

print(
    pace_model_data.loc[
        missing_fuel,
        ["driver_number", "lap_number", "fuel_effect"]
    ].to_string(index=False)
)

Missing fuel_effect after filling: 0
Filled fuel effects:
 driver_number  lap_number  fuel_effect
             2          27    -2.187172
             4          27    -2.187172
            14          27    -2.187172
            20          27    -2.187172
            22          27    -2.187172
            23          27    -2.187172
            31          26    -2.103050
            44          27    -2.187172
            63          27    -2.187172
            81          27    -2.187172
            81          28    -2.271294


In [23]:
missing_tyre = pace_model_data["tyre_degradation_effect"].isna()

pace_model_data.loc[missing_tyre, "tyre_degradation_effect"] = 0.0

print("Missing tyre degradation after filling:",
      pace_model_data["tyre_degradation_effect"].isna().sum())

print("Final feature coverage:")

print(
    pace_model_data[
        [
            "lap_duration",
            "fuel_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "tyre_degradation_effect"
        ]
    ].isna().sum()
)

Missing tyre degradation after filling: 0
Final feature coverage:
lap_duration               0
fuel_effect                0
traffic_ahead              0
traffic_behind             0
position_change            0
tyre_degradation_effect    0
dtype: int64


In [24]:
print("===== ENHANCED PACE DATA SUMMARY =====")

print("\nLap duration:")
print(pace_model_data["lap_duration"].describe())

print("\nFuel effect:")
print(pace_model_data["fuel_effect"].describe())

print("\nTyre age:")
print(pace_model_data["tyre_age"].describe())

print("\nTyre degradation effect:")
print(pace_model_data["tyre_degradation_effect"].describe())

print("\nTraffic ahead:")
print(pace_model_data["traffic_ahead"].describe())

print("\nTraffic behind:")
print(pace_model_data["traffic_behind"].describe())

print("\nPosition change:")
print(pace_model_data["position_change"].describe())

===== ENHANCED PACE DATA SUMMARY =====

Lap duration:
count    668.000000
mean      93.058708
std        3.567319
min       88.293000
25%       91.140000
50%       92.444500
75%       93.480250
max      118.710000
Name: lap_duration, dtype: float64

Fuel effect:
count    668.000000
mean      -1.917301
std        1.418689
min       -4.290221
25%       -3.533123
50%       -1.430074
75%       -0.672976
max       -0.000000
Name: fuel_effect, dtype: float64

Tyre age:
count    668.000000
mean       9.805389
std        6.804917
min        0.000000
25%        4.000000
50%        9.000000
75%       14.000000
max       27.000000
Name: tyre_age, dtype: float64

Tyre degradation effect:
count    668.000000
mean       2.110555
std        1.464720
min        0.000000
25%        0.860978
50%        1.937200
75%        3.013422
max        5.811599
Name: tyre_degradation_effect, dtype: float64

Traffic ahead:
count    668.000000
mean       0.944611
std        0.228910
min        0.000000
25%        1.

In [25]:
import statsmodels.formula.api as smf

baseline_model = smf.ols(
    "lap_duration ~ C(driver_number) + C(compound) + tyre_age",
    data=pace_model_data
).fit()

print(baseline_model.summary())

                            OLS Regression Results                            
Dep. Variable:           lap_duration   R-squared:                       0.264
Model:                            OLS   Adj. R-squared:                  0.240
Method:                 Least Squares   F-statistic:                     11.02
Date:                Sun, 27 Sep 2026   Prob (F-statistic):           2.70e-31
Time:                        16:54:32   Log-Likelihood:                -1694.6
No. Observations:                 668   AIC:                             3433.
Df Residuals:                     646   BIC:                             3532.
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 88

In [26]:
enhanced_model = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + tyre_age
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=pace_model_data
).fit()

print(enhanced_model.summary())

                            OLS Regression Results                            
Dep. Variable:           lap_duration   R-squared:                       0.339
Model:                            OLS   Adj. R-squared:                  0.313
Method:                 Least Squares   F-statistic:                     13.16
Date:                Sun, 27 Sep 2026   Prob (F-statistic):           5.47e-43
Time:                        16:58:15   Log-Likelihood:                -1658.8
No. Observations:                 668   AIC:                             3370.
Df Residuals:                     642   BIC:                             3487.
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

C:\Users\Nihan\AppData\Local\Temp\ipykernel_7072\3263284534.py:14: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()


In [27]:
train_data = pace_model_data[
    pace_model_data["lap_number"] < 45
].copy()

test_data = pace_model_data[
    pace_model_data["lap_number"] >= 45
].copy()

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

Training rows: 535
Test rows: 133


In [28]:
enhanced_model_train = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + tyre_age
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=train_data
).fit()

test_predictions = enhanced_model_train.predict(test_data)

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

test_r2 = r2_score(
    test_data["lap_duration"],
    test_predictions
)

test_rmse = np.sqrt(
    mean_squared_error(
        test_data["lap_duration"],
        test_predictions
    )
)

print("===== ENHANCED MODEL TEST PERFORMANCE =====")
print("Test R²:", test_r2)
print("Test RMSE:", test_rmse)

C:\Users\Nihan\AppData\Local\Temp\ipykernel_7072\2439155361.py:14: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()


===== ENHANCED MODEL TEST PERFORMANCE =====
Test R²: -0.598773123352653
Test RMSE: 1.6408130015000804


In [29]:
enhanced_model_v2 = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=train_data
).fit()

test_predictions_v2 = enhanced_model_v2.predict(test_data)

test_r2_v2 = r2_score(
    test_data["lap_duration"],
    test_predictions_v2
)

test_rmse_v2 = np.sqrt(
    mean_squared_error(
        test_data["lap_duration"],
        test_predictions_v2
    )
)

print("===== ENHANCED MODEL V2 =====")
print("Test R²:", test_r2_v2)
print("Test RMSE:", test_rmse_v2)

===== ENHANCED MODEL V2 =====
Test R²: -0.5987731233534486
Test RMSE: 1.6408130015004887


In [30]:
baseline_model_train = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + tyre_age
    """,
    data=train_data
).fit()

baseline_predictions = baseline_model_train.predict(test_data)

baseline_r2 = r2_score(
    test_data["lap_duration"],
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        test_data["lap_duration"],
        baseline_predictions
    )
)

print("===== BASELINE TEST PERFORMANCE =====")
print("Test R²:", baseline_r2)
print("Test RMSE:", baseline_rmse)

===== BASELINE TEST PERFORMANCE =====
Test R²: -2.8653388837679707
Test RMSE: 2.551287827797025


In [31]:
print("===== TEST LAP DISTRIBUTION =====")

print(
    test_data["lap_number"]
    .value_counts()
    .sort_index()
)

print("\n===== TEST LAP DURATION BY LAP =====")

print(
    test_data.groupby("lap_number")["lap_duration"]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

===== TEST LAP DISTRIBUTION =====
lap_number
45    18
46    18
47    18
48    17
49    18
50    18
51    14
52    12
Name: count, dtype: int64

===== TEST LAP DURATION BY LAP =====
            count    mean     min     max
lap_number                               
45             18  90.447  89.137  93.159
46             18  90.356  89.233  91.688
47             18  90.469  89.049  93.599
48             17  90.749  88.952  95.931
49             18  91.055  89.283  97.360
50             18  90.766  89.286  94.150
51             14  90.950  88.748  94.814
52             12  90.548  88.293  92.090


In [32]:
comparison = test_data[
    [
        "driver_number",
        "lap_number",
        "compound",
        "tyre_age",
        "lap_duration"
    ]
].copy()

comparison["predicted_lap_duration"] = test_predictions_v2

comparison["error"] = (
    comparison["predicted_lap_duration"]
    - comparison["lap_duration"]
)

print("===== PREDICTION COMPARISON =====")

print(
    comparison[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "predicted_lap_duration",
            "error"
        ]
    ]
    .sort_values(["lap_number", "driver_number"])
    .head(30)
    .round(3)
    .to_string(index=False)
)

print("\n===== ERROR SUMMARY =====")

print(
    comparison["error"].describe().round(3)
)

===== PREDICTION COMPARISON =====
 driver_number  lap_number compound  tyre_age  lap_duration  predicted_lap_duration  error
             1          45     HARD       6.0        89.137                  89.967  0.830
             2          45     SOFT       6.0        90.613                  90.693  0.080
             3          45     SOFT       7.0        90.955                  91.706  0.751
             4          45     SOFT       5.0        89.635                  89.635  0.000
            11          45   MEDIUM       7.0        90.565                  91.823  1.258
            14          45   MEDIUM       6.0        89.871                  91.598  1.727
            16          45     SOFT       7.0        89.911                  90.826  0.915
            18          45   MEDIUM       5.0        90.025                  90.910  0.885
            20          45     SOFT       7.0        91.722                  92.352  0.630
            22          45     SOFT       6.0        91.

In [33]:
print("===== LARGEST PREDICTION ERRORS =====")

print(
    comparison[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "predicted_lap_duration",
            "error"
        ]
    ]
    .sort_values("error")
    .head(10)
    .round(3)
    .to_string(index=False)
)

print("\n===== MOST OVERPREDICTED LAPS =====")

print(
    comparison[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "predicted_lap_duration",
            "error"
        ]
    ]
    .sort_values("error", ascending=False)
    .head(10)
    .round(3)
    .to_string(index=False)
)

===== LARGEST PREDICTION ERRORS =====
 driver_number  lap_number compound  tyre_age  lap_duration  predicted_lap_duration  error
            11          49     SOFT       1.0        97.360                  90.228 -7.132
             3          51     SOFT      13.0        94.814                  92.704 -2.110
             3          50     SOFT      12.0        94.150                  92.537 -1.613
            31          48   MEDIUM       9.0        95.931                  95.225 -0.706
            77          45     SOFT       7.0        93.159                  92.521 -0.638
             3          49     SOFT      11.0        92.926                  92.371 -0.555
            16          51     SOFT      13.0        91.674                  91.157 -0.517
             4          48     SOFT       8.0        90.556                  90.134 -0.422
            22          45     SOFT       6.0        91.438                  91.201 -0.237
             4          46     SOFT       6.0       

In [34]:
problem_drivers = [11, 31, 23, 81]

problem_context = pace_model_data[
    pace_model_data["driver_number"].isin(problem_drivers)
].copy()

print(
    problem_context[
        [
            "driver_number",
            "lap_number",
            "stint_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "fuel_effect",
            "tyre_degradation_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "position"
        ]
    ]
    .sort_values(["driver_number", "lap_number"])
    .tail(80)
    .round(3)
    .to_string(index=False)
)

 driver_number  lap_number  stint_number compound  tyre_age  lap_duration  fuel_effect  tyre_degradation_effect  traffic_ahead  traffic_behind  position_change  position
            23          44           3.0   MEDIUM       5.0        90.740       -3.617                    1.076            1.0             1.0              0.0      10.0
            23          45           3.0   MEDIUM       6.0        90.406       -3.701                    1.291            1.0             1.0              0.0      10.0
            23          46           3.0   MEDIUM       7.0        90.234       -3.785                    1.507            1.0             0.0             -1.0       9.0
            23          47           3.0   MEDIUM       8.0        90.148       -3.870                    1.722            1.0             0.0              0.0       9.0
            23          48           3.0   MEDIUM       9.0        90.068       -3.954                    1.937            1.0             0.0        

In [35]:
race_pace_by_lap = (
    pace_model_data
    .groupby("lap_number")["lap_duration"]
    .agg(
        median_lap_time="median",
        mean_lap_time="mean",
        driver_count="count"
    ).reset_index()
)
print(race_pace_by_lap.round(3).to_string(index=False))

 lap_number  median_lap_time  mean_lap_time  driver_count
          1          100.758        100.492            18
          2           92.942         92.977            19
          3           92.605         92.719            19
          4           92.968         92.748            19
          5           92.697         92.556            19
          6           92.697         92.669            19
          7           92.733         92.889            19
          8           92.798         92.859            19
          9           92.570         92.676            19
         10           92.647         92.743            19
         11           92.491         92.593            19
         12           92.427         92.513            19
         13           92.628         92.579            18
         14           92.488         92.460            19
         15           92.543         92.469            19
         16           93.178         93.266            19
         17   

In [36]:
race_pace_by_lap["rolling_median_3"] = (
    race_pace_by_lap["median_lap_time"]
    .rolling(window=3, min_periods=1)
    .median()
)

print(
    race_pace_by_lap[
        [
            "lap_number",
            "median_lap_time",
            "rolling_median_3",
            "driver_count"
        ]
    ].round(3).to_string(index=False)
)

 lap_number  median_lap_time  rolling_median_3  driver_count
          1          100.758           100.758            18
          2           92.942            96.850            19
          3           92.605            92.942            19
          4           92.968            92.942            19
          5           92.697            92.697            19
          6           92.697            92.697            19
          7           92.733            92.697            19
          8           92.798            92.733            19
          9           92.570            92.733            19
         10           92.647            92.647            19
         11           92.491            92.570            19
         12           92.427            92.491            19
         13           92.628            92.491            18
         14           92.488            92.488            19
         15           92.543            92.543            19
         16           93

In [38]:
race_pace_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & (enhanced_lap_data["compound"].isin(["SOFT","MEDIUM","HARD"]))
].copy()

race_pace_by_lap = (
    race_pace_data
    .groupby("lap_number")["lap_duration"]
    .median()
    .reset_index(name="median_lap_time")
)

full_laps = pd.DataFrame({
    "lap_number": range(
        int(race_pace_by_lap["lap_number"].min()),
        int(race_pace_by_lap["lap_number"].max())+1
    )
})
race_pace_by_lap = (
    full_laps
    .merge(race_pace_by_lap,on="lap_number",how="left")
    .sort_values("lap_number")
)

race_pace_by_lap["rolling_median_3"] = (
    race_pace_by_lap["median_lap_time"]
    .rolling(window=3, min_periods=1)
    .median()
)

print(
    race_pace_by_lap[
        ["lap_number", "median_lap_time", "rolling_median_3"]
    ]
    .round(3)
    .to_string(index=False)
)


 lap_number  median_lap_time  rolling_median_3
          1          100.758           100.758
          2           92.942            96.850
          3           92.605            92.942
          4           92.968            92.942
          5           92.697            92.697
          6           92.697            92.697
          7           92.733            92.697
          8           92.798            92.733
          9           92.570            92.733
         10           92.647            92.647
         11           92.491            92.570
         12           92.427            92.491
         13           92.628            92.491
         14           92.488            92.488
         15           92.543            92.543
         16           93.178            92.543
         17           95.014            93.178
         18           95.789            95.014
         19          104.151            95.789
         20           97.330            97.330
         21  

In [39]:
reference_pace = race_pace_by_lap.loc[
    race_pace_by_lap["lap_number"].between(5, 15),
    "median_lap_time"
].median()

print("Reference field pace:", round(reference_pace, 3), "seconds")

# Track evolution relative to that reference
race_pace_by_lap["track_evolution"] = (
    race_pace_by_lap["rolling_median_3"] - reference_pace
)

print(
    race_pace_by_lap[
        [
            "lap_number",
            "rolling_median_3",
            "track_evolution"
        ]
    ]
    .loc[
        race_pace_by_lap["lap_number"].between(39, 52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  rolling_median_3  track_evolution
         39            92.930            0.302
         40            92.321           -0.308
         41            91.712           -0.917
         42            90.649           -1.979
         43            90.164           -2.465
         44            90.164           -2.465
         45            90.211           -2.418
         46            90.211           -2.418
         47            90.218           -2.410
         48            90.218           -2.410
         49            90.365           -2.264
         50            90.516           -2.113
         51            90.732           -1.896
         52            90.516           -2.113


In [41]:
race_pace_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & enhanced_lap_data["compound"].isin(["SOFT","MEDIUM","HARD"])
].copy()

race_pace_by_lap = (
    race_pace_data
    .groupby("lap_number")["lap_duration"]
    .median()
    .reset_index(name="median_lap_time")
)

full_laps = pd.DataFrame({
    "lap_number": range(
        int(race_pace_by_lap["lap_number"].min()),
        int(race_pace_by_lap["lap_number"].max()) +1
    )
})

race_pace_by_lap = (
    full_laps
    .merge(
        race_pace_by_lap,
        on="lap_number",
        how="left"
    )
    .sort_values("lap_number")
    .reset_index(drop=True)
)

race_pace_by_lap["previous_lap_median"] = (
    race_pace_by_lap["median_lap_time"].shift(1)
)

race_pace_by_lap["recent_field_pace"] = (
    race_pace_by_lap["previous_lap_median"]
    .rolling(
        window=3,
        min_periods=1
    ).median()
)

reference_pace = race_pace_by_lap.loc[
    race_pace_by_lap["lap_number"].between(5,15),
    "median_lap_time"
].median()

print(
    "Reference field pace:",
    round(reference_pace,3),
    "seconds"
)

race_pace_by_lap["track_evolution"] = (
    race_pace_by_lap["recent_field_pace"] - reference_pace
)

print(
    race_pace_by_lap[
        [
            "lap_number",
            "median_lap_time",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .loc[
        race_pace_by_lap["lap_number"].between(35,52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  median_lap_time  recent_field_pace  track_evolution
         35              NaN                NaN              NaN
         36              NaN                NaN              NaN
         37              NaN                NaN              NaN
         38              NaN                NaN              NaN
         39           92.930                NaN              NaN
         40           91.712             92.930            0.302
         41           90.649             92.321           -0.308
         42           90.164             91.712           -0.917
         43           90.144             90.649           -1.979
         44           90.211             90.164           -2.465
         45           90.227             90.164           -2.465
         46           90.160             90.211           -2.418
         47           90.218             90.211           -2.418
         48           90.365             90.218      

In [42]:
track_evolution_features = race_pace_by_lap[
    [
        "lap_number","track_evolution"
    ]
].copy()

enhanced_lap_data = enhanced_lap_data.merge(
    track_evolution_features,
    on="lap_number",
    how="left",
    validate="many_to_one"
)
print("Enhanced dataset shape:", enhanced_lap_data.shape)

print("\nNew columns:")
print(enhanced_lap_data.columns.tolist())

print("\nTrack evolution missing values:")
print(enhanced_lap_data["track_evolution"].isna().sum())

print("\nTrack evolution summary:")
print(
    enhanced_lap_data["track_evolution"]
    .describe()
)


Enhanced dataset shape: (962, 26)

New columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution']

Track evolution missing values:
167

Track evolution summary:
count    795.000000
mean       1.813453
std        5.238170
min       -2.464500
25%       -0.307500
50%        0.104500
75%        2.385500
max       20.347500
Name: track_evolution, dtype: float64


In [43]:
race_pace_reliability = (
    race_pace_data
    .groupby("lap_number")["lap_duration"]
    .agg(
        median_lap_time = "median",
        driver_count="count"
    ).reset_index()
)

race_pace_reliability = (
    race_pace_reliability
    .merge(
        race_pace_by_lap[
            [
                "lap_number",
                "recent_field_pace",
                "track_evolution"
            ]
        ],
        on="lap_number",
        how="left"
    )
    .sort_values("lap_number")
)

print(
    race_pace_reliability[
        [
            "lap_number",
            "driver_count",
            "median_lap_time",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

 lap_number  driver_count  median_lap_time  recent_field_pace  track_evolution
          1            18          100.758                NaN              NaN
          2            19           92.942            100.758            8.130
          3            19           92.605             96.850            4.221
          4            19           92.968             92.942            0.313
          5            19           92.697             92.942            0.313
          6            19           92.697             92.697            0.069
          7            19           92.733             92.697            0.069
          8            19           92.798             92.697            0.069
          9            19           92.570             92.733            0.105
         10            19           92.647             92.733            0.105
         11            19           92.491             92.647            0.019
         12            19           92.427          

In [44]:
# ============================================
# FILTER UNRELIABLE FIELD-PACE ESTIMATES
# ============================================

MIN_DRIVERS = 5

race_pace_reliability["reliable_median_lap_time"] = (
    race_pace_reliability["median_lap_time"]
    .where(
        race_pace_reliability["driver_count"] >= MIN_DRIVERS
    )
)


# Recalculate recent field pace using only
# reliable race-lap observations
race_pace_reliability["previous_reliable_pace"] = (
    race_pace_reliability["reliable_median_lap_time"].shift(1)
)


race_pace_reliability["recent_field_pace"] = (
    race_pace_reliability["previous_reliable_pace"]
    .rolling(
        window=3,
        min_periods=1
    )
    .median()
)


# --------------------------------------------
# Reference pace
# --------------------------------------------

reference_pace = race_pace_reliability.loc[
    race_pace_reliability["lap_number"].between(5, 15),
    "reliable_median_lap_time"
].median()

print(
    "Reference field pace:",
    round(reference_pace, 3),
    "seconds"
)


# --------------------------------------------
# Track evolution
# --------------------------------------------

race_pace_reliability["track_evolution"] = (
    race_pace_reliability["recent_field_pace"]
    - reference_pace
)


# --------------------------------------------
# Inspect suspicious late-race / transition laps
# --------------------------------------------

print(
    race_pace_reliability[
        [
            "lap_number",
            "driver_count",
            "reliable_median_lap_time",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .loc[
        race_pace_reliability["lap_number"].between(20, 52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  driver_count  reliable_median_lap_time  recent_field_pace  track_evolution
         20            15                    97.330             95.789            3.160
         21            15                    93.484             97.330            4.701
         22            15                    92.495             97.330            4.701
         23            16                    92.752             93.484            0.855
         24            16                    93.792             92.752            0.123
         25            16                    95.542             92.752            0.123
         26            16                   102.870             93.792            1.164
         27             9                   108.126             95.542            2.914
         28             1                       NaN            102.870           10.242
         39             6                    92.930            105.498           12

In [45]:
# ============================================
# FINAL LEAKAGE-SAFE TRACK EVOLUTION
# ============================================

# Start from the reliable field pace
track_evolution_data = race_pace_reliability[
    [
        "lap_number",
        "reliable_median_lap_time"
    ]
].copy()


# --------------------------------------------
# Previous-lap information only
# --------------------------------------------

track_evolution_data["previous_pace"] = (
    track_evolution_data["reliable_median_lap_time"]
    .shift(1)
)


# --------------------------------------------
# Require 3 consecutive previous race laps
# --------------------------------------------
#
# Because the race timeline contains every lap,
# NaNs represent missing/unusable race laps.
#
# min_periods=3 prevents pandas from skipping
# those missing laps and reaching far backwards.

track_evolution_data["recent_field_pace"] = (
    track_evolution_data["previous_pace"]
    .rolling(
        window=3,
        min_periods=3
    )
    .median()
)


# --------------------------------------------
# Reference early-race pace
# --------------------------------------------

reference_pace = track_evolution_data.loc[
    track_evolution_data["lap_number"].between(5, 15),
    "reliable_median_lap_time"
].median()

print(
    "Reference field pace:",
    round(reference_pace, 3),
    "seconds"
)


# --------------------------------------------
# Track evolution
# --------------------------------------------

track_evolution_data["track_evolution"] = (
    track_evolution_data["recent_field_pace"]
    - reference_pace
)


# --------------------------------------------
# Inspect transition around lap 39+
# --------------------------------------------

print(
    track_evolution_data[
        [
            "lap_number",
            "reliable_median_lap_time",
            "previous_pace",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .loc[
        track_evolution_data["lap_number"].between(35, 52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  reliable_median_lap_time  previous_pace  recent_field_pace  track_evolution
         39                    92.930            NaN                NaN              NaN
         40                    91.712         92.930                NaN              NaN
         41                    90.649         91.712                NaN              NaN
         42                    90.164         90.649             91.712           -0.917
         43                    90.144         90.164             90.649           -1.979
         44                    90.211         90.144             90.164           -2.465
         45                    90.227         90.211             90.164           -2.465
         46                    90.160         90.227             90.211           -2.418
         47                    90.218         90.160             90.211           -2.418
         48                    90.365         90.218             90.218  

In [46]:
enhanced_lap_data = enhanced_lap_data.drop(
    columns=["track_evolution"]
)
track_evolution_features = track_evolution_data[
    [
        "lap_number","track_evolution"
    ]
].copy()

enhanced_lap_data = enhanced_lap_data.merge(
    track_evolution_features,
    on="lap_number",
    how="left",
    validate="many_to_one"
)

print("Enhanced dataset shape:", enhanced_lap_data.shape)
print("\nTrack evolution missing values:")
print(
    enhanced_lap_data["track_evolution"].isna().sum()
)

print("\nTrack evolution summary:")
print(
    enhanced_lap_data["track_evolution"]
    .describe()
)

Enhanced dataset shape: (962, 26)

Track evolution missing values:
298

Track evolution summary:
count    664.000000
mean       0.279445
std        2.522388
min       -2.464500
25%       -1.896000
50%        0.068500
75%        0.549500
max       10.242000
Name: track_evolution, dtype: float64


In [47]:
pace_model_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & enhanced_lap_data["compound"].isin(
        ["SOFT", "MEDIUM", "HARD"]
    )
    & (~enhanced_lap_data["lap_number"].isin([19, 25]))
].copy()


print("Pace model data shape:", pace_model_data.shape)

print("\nMissing values:")
print(
    pace_model_data[
        [
            "fuel_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "tyre_degradation_effect",
            "track_evolution"
        ]
    ]
    .isna()
    .sum()
)

Pace model data shape: (668, 26)

Missing values:
fuel_effect                11
traffic_ahead               0
traffic_behind              0
position_change             0
tyre_degradation_effect    18
track_evolution            94
dtype: int64


In [48]:
pace_model_data["track_evolution_available"] = (
    pace_model_data["track_evolution"].notna().astype(int)
)

pace_model_data["track_evolution"] = (
    pace_model_data["track_evolution"].fillna(0.0)
)

print("Missing track evolution:")

print(
    pace_model_data["track_evolution"].isna().sum()
)

print("Availability counts:")
print(
    pace_model_data["track_evolution_available"]
    .value_counts()
    .sort_index()
)

print("Track evolution summary:")
print(
    pace_model_data["track_evolution"]
    .describe()
)

Missing track evolution:
0
Availability counts:
track_evolution_available
0     94
1    574
Name: count, dtype: int64
Track evolution summary:
count    668.000000
mean      -0.191059
std        1.701435
min       -2.464500
25%       -1.896000
50%        0.000000
75%        0.104500
max       10.242000
Name: track_evolution, dtype: float64


In [51]:
print(
    pace_model_data[
        [
            "lap_number",
            "track_evolution",
            "track_evolution_available"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "track_evolution",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

 lap_number  track_evolution  track_evolution_available
         28          10.2420                          1
         22           4.7015                          1
         21           4.7015                          1
         20           3.1605                          1
         27           2.9140                          1
         26           1.1640                          1
         23           0.8555                          1
         18           0.5495                          1
          5           0.3135                          1
          4           0.3135                          1
         24           0.1230                          1
          9           0.1045                          1
         10           0.1045                          1
          6           0.0685                          1
          7           0.0685                          1


In [53]:
train_data = pace_model_data[
    pace_model_data["lap_number"] < 45
].copy()

test_data = pace_model_data[
    pace_model_data["lap_number"] >= 45
].copy()


print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

model_without_track = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=train_data
).fit()

pred_without_track = model_without_track.predict(
    test_data
)

model_with_track = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    + track_evolution
    + track_evolution_available
    """,
    data=train_data
).fit()


pred_with_track = model_with_track.predict(
    test_data
)

rmse_without_track = np.sqrt(
    np.mean(
        (test_data["lap_duration"] - pred_without_track) ** 2
    )
)

rmse_with_track = np.sqrt(
    np.mean(
        (test_data["lap_duration"] - pred_with_track) ** 2
    )
)
r2_without_track = (
    1
    - (
        np.sum(
            (test_data["lap_duration"] - pred_without_track) ** 2
        )
        /
        np.sum(
            (test_data["lap_duration"] - test_data["lap_duration"].mean()) ** 2
        )
    )
)
r2_with_track = (
    1
    - (
        np.sum(
            (test_data["lap_duration"] - pred_with_track) ** 2
        )
        /
        np.sum(
            (test_data["lap_duration"] - test_data["lap_duration"].mean()) ** 2
        )
    )
)
comparison_results = pd.DataFrame({
    "model": [
        "Without track evolution",
        "With track evolution"
    ],
    "RMSE": [
        rmse_without_track,
        rmse_with_track
    ],
    "R2": [
        r2_without_track,
        r2_with_track
    ]
})


print(
    comparison_results.round(4).to_string(index=False)
)

Training rows: 535
Test rows: 133
                  model   RMSE      R2
Without track evolution 1.4461 -0.2418
   With track evolution 1.3247 -0.0420


In [57]:
from pathlib import Path

project = Path.cwd()

print("Notebook is running from:")
print(project)

print("\nContents of current directory:")
for item in project.iterdir():
    print(item)

print("\nExpected weather path:")
weather_path = project / "data" / "processed" / "9558" / "weather.parquet"
print(weather_path)

print("\nDoes expected path exist?")
print(weather_path.exists())

print("\nDoes data/processed/9558 exist?")
print((project / "data" / "processed" / "9558").exists())

if (project / "data" / "processed" / "9558").exists():
    print("\nContents of data/processed/9558:")
    for item in (project / "data" / "processed" / "9558").iterdir():
        print(item.name)

Notebook is running from:
c:\Users\Nihan\.vscode\pitwall\notebooks

Contents of current directory:
c:\Users\Nihan\.vscode\pitwall\notebooks\01_data_exploration.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\02_race_analysis.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\03_traffic_features.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\04_fuel_effects.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\05_tyre_degradation.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\06_enhanced_pace_model.ipynb

Expected weather path:
c:\Users\Nihan\.vscode\pitwall\notebooks\data\processed\9558\weather.parquet

Does expected path exist?
False

Does data/processed/9558 exist?
False


In [58]:
import pandas as pd

weather_path = "../data/processed/9558/weather.parquet"

weather = pd.read_parquet(weather_path)

print("Shape:", weather.shape)

print("\nColumns:")
print(weather.columns.tolist())

print("\nFirst 5 rows:")
display(weather.head())

print("\nDtypes:")
print(weather.dtypes)

print("\nMissing values:")
print(weather.isna().sum())

date_cols = [
    col for col in weather.columns
    if "date" in col.lower() or "time" in col.lower()
]

print("\nTimestamp/date columns:", date_cols)

for col in date_cols:
    dt = pd.to_datetime(weather[col])

    print(f"\n{col}:")
    print("  Minimum:", dt.min())
    print("  Maximum:", dt.max())
    print("  Unique values:", dt.nunique())

    if len(dt) > 1:
        print("\n  Most common time intervals:")
        print(dt.sort_values().diff().value_counts().head(10))

Shape: (147, 10)

Columns:
['date', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction']

First 5 rows:


,date,session_key,meeting_key,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
0,2024-07-07 13:05:02.119000+00:00,9558,1240,28.1,2.2,0,67.0,991.8,15.9,226
1,2024-07-07 13:06:02.098000+00:00,9558,1240,27.6,1.5,0,66.0,991.7,15.9,268
2,2024-07-07 13:07:02.122000+00:00,9558,1240,27.0,1.4,0,65.0,991.7,15.9,291
3,2024-07-07 13:08:02.127000+00:00,9558,1240,27.0,1.1,0,66.0,991.8,15.9,195
4,2024-07-07 13:09:02.125000+00:00,9558,1240,31.0,1.9,0,64.0,991.8,15.9,225



Dtypes:
date                 datetime64[us, UTC]
session_key                        int64
meeting_key                        int64
track_temperature                float64
wind_speed                       float64
rainfall                           int64
humidity                         float64
pressure                         float64
air_temperature                  float64
wind_direction                     int64
dtype: object

Missing values:
date                 0
session_key          0
meeting_key          0
track_temperature    0
wind_speed           0
rainfall             0
humidity             0
pressure             0
air_temperature      0
wind_direction       0
dtype: int64

Timestamp/date columns: ['date']

date:
  Minimum: 2024-07-07 13:05:02.119000+00:00
  Maximum: 2024-07-07 15:31:03.204000+00:00
  Unique values: 147

  Most common time intervals:
date
0 days 00:01:00.020000    7
0 days 00:01:00.005000    6
0 days 00:01:00.014000    6
0 days 00:01:00           6
0 days 00

In [59]:
print("lap data enhanced shape:", enhanced_lap_data.shape)
print("Columns containing date/time")
print([
    col for col  in enhanced_lap_data.columns
    if "date" in col.lower() or "time" in col.lower()
])

print("First 5 lap records:")
display(
    enhanced_lap_data[
        [col for col in enhanced_lap_data.columns
        if "date" in col.lower() or "time" in col.lower()]
        +["lap_number"]
    ].head()
)

lap data enhanced shape: (962, 26)
Columns containing date/time
['date_start']
First 5 lap records:


,date_start,lap_number
0,2024-07-07 14:03:12.540000+00:00,1
1,2024-07-07 14:04:49.750000+00:00,2
2,2024-07-07 14:06:21.461000+00:00,3
3,2024-07-07 14:07:53.675000+00:00,4
4,2024-07-07 14:09:25.527000+00:00,5


In [61]:
weather_check = weather[
    ["date", "session_key"]
].sort_values(["session_key", "date"]).copy()

lap_check = enhanced_lap_data[
    ["date_start", "session_key", "lap_number"]
].sort_values(["session_key", "date_start"]).copy()

aligned_check = pd.merge_asof(
    lap_check,
    weather_check,
    left_on="date_start",
    right_on="date",
    by="session_key",
    direction="backward"
)

aligned_check["weather_lag_seconds"] = (
    aligned_check["date_start"] - aligned_check["date"]
).dt.total_seconds()

print("Weather alignment lag:")
print(aligned_check["weather_lag_seconds"].describe())

print("\nMaximum lag:", aligned_check["weather_lag_seconds"].max(), "seconds")

print("\nFirst 10 aligned laps:")
display(
    aligned_check[
        ["session_key", "lap_number", "date_start", "date",
         "weather_lag_seconds"]
    ].head(10)
)

ValueError: Merge keys contain null values on left side

In [62]:

print("Total laps:", len(enhanced_lap_data))

print("\nNull date_start values:")
print(enhanced_lap_data["date_start"].isna().sum())

print("\nRows with null date_start:")
display(
    enhanced_lap_data[
        enhanced_lap_data["date_start"].isna()
    ]
)

print("\nNull session_key values:")
print(enhanced_lap_data["session_key"].isna().sum())

Total laps: 962

Null date_start values:
1

Rows with null date_start:


,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,...,stint_number,compound,tyre_age,fuel_effect,position,traffic_ahead,traffic_behind,position_change,tyre_degradation_effect,track_evolution
208,1240,9558,11,1,NaT,NaN,39.507,26.165,288.0,252.0,...,1.0,HARD,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Null session_key values:
0


In [63]:
lap_weather = enhanced_lap_data[
    ["meeting_key", "session_key","driver_number","lap_number","date_start"]
].copy(0)

lap_weather_valid = lap_weather.dropna(subset=["date_start"]).copy()

lap_weather_valid = lap_weather.dropna(subset=["date_start"]).copy()

weather_alignment = weather[
    ["session_key", "date",
     "track_temperature",
     "wind_speed",
     "rainfall",
     "humidity",
     "pressure",
     "air_temperature",
     "wind_direction"]
].copy()

lap_weather_valid["date_start"] = pd.to_datetime(
    lap_weather_valid["date_start"], utc=True
)

weather_alignment["date"] = pd.to_datetime(
    weather_alignment["date"], utc=True
)

lap_weather_valid = lap_weather_valid.sort_values(
    ["session_key", "date_start"]
)

weather_alignment = weather_alignment.sort_values(
    ["session_key", "date"]
)

print("Valid lap rows:", len(lap_weather_valid))
print("Weather rows:", len(weather_alignment))

print("\nLap sessions:")
print(lap_weather_valid["session_key"].unique())

print("\nWeather sessions:")
print(weather_alignment["session_key"].unique())

Valid lap rows: 961
Weather rows: 147

Lap sessions:
[9558]

Weather sessions:
[9558]


In [64]:
lap_weather_aligned = pd.merge_asof(
   lap_weather_valid,
   weather_alignment,
   left_on="date_start",
   right_on="date",
   by="session_key",
   direction="backward" 
)

lap_weather_aligned["weather_lag_seconds"] = (
    lap_weather_aligned["date_start"]
    - lap_weather_aligned["date"]
).dt.total_seconds()

print("Aligned shape:", lap_weather_aligned.shape)

print("\nWeather lag statistics:")
print(lap_weather_aligned["weather_lag_seconds"].describe())

print("\nMaximum weather lag:",
      lap_weather_aligned["weather_lag_seconds"].max(),
      "seconds")

print("\nFirst 10 aligned laps:")
display(
    lap_weather_aligned[
        ["lap_number", "date_start", "date",
         "weather_lag_seconds",
         "track_temperature",
         "air_temperature",
         "rainfall"]
    ].head(10)
)

Aligned shape: (961, 14)

Weather lag statistics:
count    961.000000
mean      29.377612
std       17.568594
min        0.019000
25%       13.337000
50%       29.296000
75%       44.449000
max       59.981000
Name: weather_lag_seconds, dtype: float64

Maximum weather lag: 59.981 seconds

First 10 aligned laps:


,lap_number,date_start,date,weather_lag_seconds,track_temperature,air_temperature,rainfall
0,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
1,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
2,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
3,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
4,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
5,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
6,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
7,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
8,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
9,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0


In [65]:
weather_features = [
    "track_temperature",
    "wind_speed",
    "rainfall",
    "humidity",
    "pressure",
    "air_temperature",
    "wind_direction"
]

print("Unique values and ranges:")

for col in weather_features:
    print(f"{col}:")
    print(f"  Unique values: {weather[col].nunique()}")
    print(f"  Min: {weather[col].min()}")
    print(f"  Max: {weather[col].max()}")
    
    if weather[col].nunique() <= 15:
        print(f"  Values: {sorted(weather[col].unique())}")
    
    print()

Unique values and ranges:
track_temperature:
  Unique values: 68
  Min: 20.7
  Max: 37.9

wind_speed:
  Unique values: 21
  Min: 0.4
  Max: 2.5

rainfall:
  Unique values: 2
  Min: 0
  Max: 1
  Values: [np.int64(0), np.int64(1)]

humidity:
  Unique values: 20
  Min: 59.0
  Max: 78.0

pressure:
  Unique values: 10
  Min: 991.7
  Max: 992.6
  Values: [np.float64(991.7), np.float64(991.8), np.float64(991.9), np.float64(992.0), np.float64(992.1), np.float64(992.2), np.float64(992.3), np.float64(992.4), np.float64(992.5), np.float64(992.6)]

air_temperature:
  Unique values: 24
  Min: 14.5
  Max: 16.8

wind_direction:
  Unique values: 87
  Min: 0
  Max: 335



In [66]:
print("Rainfall observations:")

display(
    weather.loc[
        weather["rainfall"] == 1,
        ["date", "track_temperature", "air_temperature", "humidity", "wind_speed", "rainfall"]
    ]
)
print("\nRainfall count:")
print(weather["rainfall"].value_counts())

print("\nWeather observations over time:")
display(
    weather[
        ["date", "track_temperature", "wind_speed",
         "rainfall", "humidity", "pressure",
         "air_temperature", "wind_direction"]
    ].head(20)
)

Rainfall observations:


,date,track_temperature,air_temperature,humidity,wind_speed,rainfall
5,2024-07-07 13:10:02.120000+00:00,31.3,15.9,64.0,2.0,1
6,2024-07-07 13:11:02.102000+00:00,31.3,15.8,64.0,2.0,1
7,2024-07-07 13:12:02.096000+00:00,31.0,15.8,67.0,1.1,1
8,2024-07-07 13:13:02.111000+00:00,26.5,15.8,68.0,1.0,1
9,2024-07-07 13:14:02.125000+00:00,26.5,15.8,70.0,0.6,1
10,2024-07-07 13:15:02.145000+00:00,26.6,15.7,70.0,0.7,1
11,2024-07-07 13:16:02.153000+00:00,26.5,15.6,70.0,0.7,1
12,2024-07-07 13:17:02.146000+00:00,26.5,15.5,71.0,0.7,1
13,2024-07-07 13:18:02.160000+00:00,27.6,15.4,75.0,1.0,1
14,2024-07-07 13:19:02.145000+00:00,29.4,15.4,73.0,0.6,1



Rainfall count:
rainfall
0    96
1    51
Name: count, dtype: int64

Weather observations over time:


,date,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
0,2024-07-07 13:05:02.119000+00:00,28.1,2.2,0,67.0,991.8,15.9,226
1,2024-07-07 13:06:02.098000+00:00,27.6,1.5,0,66.0,991.7,15.9,268
2,2024-07-07 13:07:02.122000+00:00,27.0,1.4,0,65.0,991.7,15.9,291
3,2024-07-07 13:08:02.127000+00:00,27.0,1.1,0,66.0,991.8,15.9,195
4,2024-07-07 13:09:02.125000+00:00,31.0,1.9,0,64.0,991.8,15.9,225
5,2024-07-07 13:10:02.120000+00:00,31.3,2.0,1,64.0,991.9,15.9,206
6,2024-07-07 13:11:02.102000+00:00,31.3,2.0,1,64.0,991.9,15.8,206
7,2024-07-07 13:12:02.096000+00:00,31.0,1.1,1,67.0,991.8,15.8,242
8,2024-07-07 13:13:02.111000+00:00,26.5,1.0,1,68.0,991.8,15.8,254
9,2024-07-07 13:14:02.125000+00:00,26.5,0.6,1,70.0,991.8,15.8,0


In [68]:
print("Weather time range:")
print("Start:", weather["date"].min())
print("End:",weather["date"].max())

print("\nWeather observations around race start:")

race_start = enhanced_lap_data["date_start"].min()

display(
    weather[
        (weather["date"] >= race_start - pd.Timedelta(minutes=5)) &
        (weather["date"] <= race_start + pd.Timedelta(minutes=5))
    ]
)

print("Weather observations around race end:")

race_end = enhanced_lap_data["date_start"].max()

display(
    weather[
        (weather["date"] >= race_end - pd.Timedelta(minutes=5)) &
        (weather["date"] <= race_end + pd.Timedelta(minutes=5))
    ]
)

Weather time range:
Start: 2024-07-07 13:05:02.119000+00:00
End: 2024-07-07 15:31:03.204000+00:00

Weather observations around race start:


,date,session_key,meeting_key,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
54,2024-07-07 13:59:02.452000+00:00,9558,1240,33.0,1.7,0,64.0,992.0,16.1,243
55,2024-07-07 14:00:02.446000+00:00,9558,1240,33.0,1.8,0,63.0,992.0,16.1,238
56,2024-07-07 14:01:02.472000+00:00,9558,1240,33.0,2.0,0,64.0,992.0,16.1,233
57,2024-07-07 14:02:02.478000+00:00,9558,1240,33.0,1.9,0,63.0,992.0,16.1,254
58,2024-07-07 14:03:02.496000+00:00,9558,1240,33.0,1.6,0,64.0,992.0,16.1,275
59,2024-07-07 14:04:02.504000+00:00,9558,1240,33.0,1.6,0,63.0,992.0,16.2,215
60,2024-07-07 14:05:02.495000+00:00,9558,1240,33.0,1.8,0,61.0,991.9,16.2,249
61,2024-07-07 14:06:02.504000+00:00,9558,1240,33.0,2.0,0,62.0,991.9,16.3,235
62,2024-07-07 14:07:02.521000+00:00,9558,1240,24.3,1.9,0,61.0,992.0,16.3,237
63,2024-07-07 14:08:02.584000+00:00,9558,1240,24.1,2.5,0,61.0,991.9,16.3,216


Weather observations around race end:


,date,session_key,meeting_key,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
136,2024-07-07 15:21:03.104000+00:00,9558,1240,25.2,1.8,0,69.0,992.4,15.8,222
137,2024-07-07 15:22:03.113000+00:00,9558,1240,25.4,1.7,0,69.0,992.5,15.8,219
138,2024-07-07 15:23:03.118000+00:00,9558,1240,25.5,1.4,0,69.0,992.5,15.9,230
139,2024-07-07 15:24:03.133000+00:00,9558,1240,25.3,1.2,0,68.0,992.4,15.9,258
140,2024-07-07 15:25:03.120000+00:00,9558,1240,25.2,0.7,0,69.0,992.5,15.8,241
141,2024-07-07 15:26:03.163000+00:00,9558,1240,24.9,1.9,0,68.0,992.5,15.8,230
142,2024-07-07 15:27:03.155000+00:00,9558,1240,24.8,2.1,0,69.0,992.4,15.8,224
143,2024-07-07 15:28:03.175000+00:00,9558,1240,25.0,1.7,0,70.0,992.4,15.9,261
144,2024-07-07 15:29:03.195000+00:00,9558,1240,26.1,1.0,0,71.0,992.4,15.9,222
145,2024-07-07 15:30:03.198000+00:00,9558,1240,26.4,1.6,0,69.0,992.5,16.0,230


In [69]:
# Inspect unusually large minute-to-minute weather changes

weather_sorted = weather.sort_values("date").copy()

numeric_weather = [
    "track_temperature",
    "wind_speed",
    "humidity",
    "pressure",
    "air_temperature",
    "wind_direction"
]

for col in numeric_weather:
    weather_sorted[f"{col}_change"] = weather_sorted[col].diff()

print("Largest absolute changes between consecutive weather observations:\n")

for col in numeric_weather:
    change_col = f"{col}_change"

    idx = weather_sorted[change_col].abs().idxmax()
    row = weather_sorted.loc[idx]

    print(f"{col}:")
    print(f"  Change: {row[change_col]}")
    print(f"  Time:   {row['date']}")
    print()

print("Rainfall transitions:")

rain_change = weather_sorted["rainfall"].diff()
display(
    weather_sorted.loc[
        rain_change.fillna(0) != 0,
        ["date", "rainfall", "track_temperature",
         "air_temperature", "humidity", "wind_speed"]
    ]
)

Largest absolute changes between consecutive weather observations:

track_temperature:
  Change: -8.7
  Time:   2024-07-07 14:07:02.521000+00:00

wind_speed:
  Change: 1.6
  Time:   2024-07-07 13:35:02.291000+00:00

humidity:
  Change: -5.0
  Time:   2024-07-07 13:29:02.220000+00:00

pressure:
  Change: 0.20000000000004547
  Time:   2024-07-07 13:56:02.427000+00:00

air_temperature:
  Change: -0.3000000000000007
  Time:   2024-07-07 14:35:02.692000+00:00

wind_direction:
  Change: 327.0
  Time:   2024-07-07 13:16:02.153000+00:00

Rainfall transitions:


,date,rainfall,track_temperature,air_temperature,humidity,wind_speed
5,2024-07-07 13:10:02.120000+00:00,1,31.3,15.9,64.0,2.0
17,2024-07-07 13:22:02.173000+00:00,0,32.9,15.3,72.0,1.0
21,2024-07-07 13:26:02.189000+00:00,1,34.5,15.8,71.0,0.8
23,2024-07-07 13:28:02.230000+00:00,0,36.9,16.0,71.0,1.0
85,2024-07-07 14:30:02.680000+00:00,1,23.9,16.6,60.0,1.6
120,2024-07-07 15:05:02.943000+00:00,0,24.6,14.6,78.0,1.4
128,2024-07-07 15:13:03.028000+00:00,1,24.5,15.4,72.0,2.0
130,2024-07-07 15:15:03.048000+00:00,0,25.0,15.5,73.0,2.0


In [70]:
print("Lap level weather summary:")

weather_features = [
    "track_temperature",
    "wind_speed",
    "rainfall",
    "humidity",
    "pressure",
    "air_temperature",
    "wind_direction",
    "weather_lag_seconds"
]

display(
    lap_weather_aligned[weather_features].describe()
)
print("\nRainfall at lap start:")
display(
    lap_weather_aligned["rainfall"].value_counts(dropna=False)
)

print("Laps aligned to rainfall:")

display(
    lap_weather_aligned.loc[
        lap_weather_aligned["rainfall"] == 1,
        [
            "driver_number",
            "lap_number",
            "date_start",
            "date",
            "rainfall",
            "track_temperature",
            "air_temperature",
            "humidity",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].drop_duplicates(
        subset=["lap_number", "date"]
    ).sort_values("date")
)

print("Track temperature around 14:07 anomaly:")
display(
    weather.loc[
        (weather["date"] >= "2024-07-07 14:04:00+00:00") &
        (weather["date"] <= "2024-07-07 14:10:00+00:00"),
        [
            "date",
            "track_temperature",
            "air_temperature",
            "humidity",
            "wind_speed",
            "rainfall"
        ]
    ]
)

Lap level weather summary:


,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction,weather_lag_seconds
count,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000
mean,24.125494,1.483039,0.423517,67.522373,992.246410,15.651301,232.952133,29.377612
std,2.796364,0.470786,0.494373,5.822708,0.251806,0.728670,40.770442,17.568594
min,20.700000,0.500000,0.000000,59.000000,991.800000,14.500000,59.000000,0.019000
25%,22.000000,1.100000,0.000000,61.000000,992.000000,15.100000,219.000000,13.337000
50%,24.300000,1.500000,0.000000,68.000000,992.200000,15.600000,237.000000,29.296000
75%,25.200000,1.900000,1.000000,73.000000,992.500000,16.300000,251.000000,44.449000
max,33.000000,2.500000,1.000000,78.000000,992.600000,16.800000,335.000000,59.981000



Rainfall at lap start:


rainfall
0    554
1    407
Name: count, dtype: int64

Laps aligned to rainfall:


,driver_number,lap_number,date_start,date,rainfall,track_temperature,air_temperature,humidity,wind_speed,weather_lag_seconds
340,31,18,2024-07-07 14:30:08.240000+00:00,2024-07-07 14:30:02.680000+00:00,1,23.9,16.6,60.0,1.6,5.560
342,44,19,2024-07-07 14:30:52.867000+00:00,2024-07-07 14:30:02.680000+00:00,1,23.9,16.6,60.0,1.6,50.187
347,55,19,2024-07-07 14:31:03.691000+00:00,2024-07-07 14:31:02.667000+00:00,1,23.2,16.4,63.0,1.5,1.024
360,24,19,2024-07-07 14:32:06.641000+00:00,2024-07-07 14:32:02.688000+00:00,1,22.9,16.2,64.0,1.5,3.953
361,44,20,2024-07-07 14:32:36.572000+00:00,2024-07-07 14:32:02.688000+00:00,1,22.9,16.2,64.0,1.5,33.884
...,...,...,...,...,...,...,...,...,...,...
811,18,44,2024-07-07 15:13:06.495000+00:00,2024-07-07 15:13:03.028000+00:00,1,24.5,15.4,72.0,2.0,3.467
820,44,45,2024-07-07 15:13:42.570000+00:00,2024-07-07 15:13:03.028000+00:00,1,24.5,15.4,72.0,2.0,39.542
825,55,45,2024-07-07 15:14:09.436000+00:00,2024-07-07 15:14:03.040000+00:00,1,24.5,15.4,73.0,2.0,6.396
824,77,44,2024-07-07 15:14:06.347000+00:00,2024-07-07 15:14:03.040000+00:00,1,24.5,15.4,73.0,2.0,3.307


Track temperature around 14:07 anomaly:


,date,track_temperature,air_temperature,humidity,wind_speed,rainfall
59,2024-07-07 14:04:02.504000+00:00,33.0,16.2,63.0,1.6,0
60,2024-07-07 14:05:02.495000+00:00,33.0,16.2,61.0,1.8,0
61,2024-07-07 14:06:02.504000+00:00,33.0,16.3,62.0,2.0,0
62,2024-07-07 14:07:02.521000+00:00,24.3,16.3,61.0,1.9,0
63,2024-07-07 14:08:02.584000+00:00,24.1,16.3,61.0,2.5,0
64,2024-07-07 14:09:02.537000+00:00,24.0,16.2,60.0,2.2,0


In [71]:
# Inspect relationship between weather conditions and lap duration

weather_pace = lap_weather_aligned.merge(
    enhanced_lap_data[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound"
        ]
    ],
    on=["driver_number", "lap_number"],
    how="left"
)

print("Weather + pace shape:", weather_pace.shape)

print("\nLap duration by rainfall:")
display(
    weather_pace.groupby("rainfall")["lap_duration"].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
)

print("\nLap duration by rainfall and compound:")
display(
    weather_pace.groupby(["rainfall", "compound"])["lap_duration"].agg(
        ["count", "mean", "median", "std"]
    )
)

print("\nLap duration around track-temperature anomaly:")

display(
    weather_pace.loc[
        (weather_pace["date_start"] >= "2024-07-07 14:05:00+00:00") &
        (weather_pace["date_start"] <= "2024-07-09 14:10:00+00:00"),
        [
            "driver_number",
            "lap_number",
            "date_start",
            "lap_duration",
            "compound",
            "track_temperature",
            "rainfall"
        ]
    ].sort_values("date_start").head(40)
)

Weather + pace shape: (961, 16)

Lap duration by rainfall:


,count,mean,median,std,min,max
rainfall,,,,,,
0,553,93.088911,92.361,4.686977,88.293,126.729
1,406,101.933342,102.441,7.868429,89.137,128.459



Lap duration by rainfall and compound:


count        mean    median        std
rainfall compound                                            
0        HARD             39   91.766179   90.0560   4.509557
         INTERMEDIATE      1   99.440000   99.4400        NaN
         MEDIUM          359   93.208312   92.5900   3.933854
         SOFT            154   93.104305   91.3210   6.103452
1        HARD              4   99.726250   95.7285  13.721954
         INTERMEDIATE    233  105.011146  103.2290   6.497258
         MEDIUM          149   97.521443   95.2350   6.298948
         SOFT             20   99.387000   91.3095  13.740177


Lap duration around track-temperature anomaly:


,driver_number,lap_number,date_start,lap_duration,compound,track_temperature,rainfall
37,11,2,2024-07-07 14:05:02.418000+00:00,92.942,HARD,33.0,0
38,63,3,2024-07-07 14:06:19.683000+00:00,91.298,MEDIUM,33.0,0
39,44,3,2024-07-07 14:06:20.512000+00:00,91.716,MEDIUM,33.0,0
40,1,3,2024-07-07 14:06:21.461000+00:00,92.200,MEDIUM,33.0,0
41,4,3,2024-07-07 14:06:22.424000+00:00,91.991,MEDIUM,33.0,0
42,81,3,2024-07-07 14:06:23.337000+00:00,91.927,MEDIUM,33.0,0
43,55,3,2024-07-07 14:06:24.036000+00:00,92.102,MEDIUM,33.0,0
44,18,3,2024-07-07 14:06:24.689000+00:00,92.307,MEDIUM,33.0,0
45,16,3,2024-07-07 14:06:25.521000+00:00,92.369,MEDIUM,33.0,0
46,27,3,2024-07-07 14:06:25.948000+00:00,92.605,MEDIUM,33.0,0


In [72]:
# Check candidate weather features before adding them to the pace model

candidate_weather = [
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed"
]

print("Candidate weather feature statistics:\n")

display(
    lap_weather_aligned[candidate_weather].describe()
)

print("\nUnique values:\n")

for col in candidate_weather:
    print(f"{col}: {lap_weather_aligned[col].nunique()} unique values")

print("\nCorrelation matrix:\n")

display(
    lap_weather_aligned[candidate_weather].corr()
)

print("\nWeather conditions by race lap:")

display(
    lap_weather_aligned.groupby("lap_number")[candidate_weather]
    .first()
    .head(25)
)

Candidate weather feature statistics:



,rainfall,humidity,air_temperature,wind_speed
count,961.000000,961.000000,961.000000,961.000000
mean,0.423517,67.522373,15.651301,1.483039
std,0.494373,5.822708,0.728670,0.470786
min,0.000000,59.000000,14.500000,0.500000
25%,0.000000,61.000000,15.100000,1.100000
50%,0.000000,68.000000,15.600000,1.500000
75%,1.000000,73.000000,16.300000,1.900000
max,1.000000,78.000000,16.800000,2.500000



Unique values:

rainfall: 2 unique values
humidity: 18 unique values
air_temperature: 23 unique values
wind_speed: 20 unique values

Correlation matrix:



,rainfall,humidity,air_temperature,wind_speed
rainfall,1.000000,0.464418,-0.664438,-0.425167
humidity,0.464418,1.000000,-0.904431,-0.379536
air_temperature,-0.664438,-0.904431,1.000000,0.462054
wind_speed,-0.425167,-0.379536,0.462054,1.000000



Weather conditions by race lap:


,rainfall,humidity,air_temperature,wind_speed
lap_number,,,,
1,0,64.0,16.1,1.6
2,0,63.0,16.2,1.6
3,0,62.0,16.3,2.0
4,0,61.0,16.3,1.9
5,0,60.0,16.2,2.2
6,0,60.0,16.3,2.4
7,0,61.0,16.3,2.1
8,0,60.0,16.4,2.4
9,0,61.0,16.5,2.0


In [73]:
weather_features = lap_weather_aligned[
    [
        "driver_number",
        "lap_number",
        "rainfall",
        "humidity",
        "air_temperature",
        "wind_speed",
        "weather_lag_seconds"
    ]
].copy()

print("Weather feature rows:", len(weather_features))
print("Unique driver-lap pairs:",
      weather_features[["driver_number", "lap_number"]].drop_duplicates().shape[0])

print("\nDuplicate driver-lap pairs:")
print(
    weather_features.duplicated(
        subset=["driver_number", "lap_number"]
    ).sum()
)

Weather feature rows: 961
Unique driver-lap pairs: 961

Duplicate driver-lap pairs:
0


In [74]:
enhanced_lap_data_weather = enhanced_lap_data.merge(
    weather_features,
    on=["driver_number", "lap_number"],
    how="left",
    validate="one_to_one"
)

print("Original shape:", enhanced_lap_data.shape)
print("New shape:", enhanced_lap_data_weather.shape)

print("\nWeather feature missing values:")
display(
    enhanced_lap_data_weather[
        [
            "rainfall",
            "humidity",
            "air_temperature",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].isna().sum()
)

print("\nWeather columns:")
display(
    enhanced_lap_data_weather[
        [
            "driver_number",
            "lap_number",
            "date_start",
            "rainfall",
            "humidity",
            "air_temperature",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].head(10)
)

Original shape: (962, 26)
New shape: (962, 31)

Weather feature missing values:


rainfall               1
humidity               1
air_temperature        1
wind_speed             1
weather_lag_seconds    1
dtype: int64


Weather columns:


,driver_number,lap_number,date_start,rainfall,humidity,air_temperature,wind_speed,weather_lag_seconds
0,1,1,2024-07-07 14:03:12.540000+00:00,0.0,64.0,16.1,1.6,10.044
1,1,2,2024-07-07 14:04:49.750000+00:00,0.0,63.0,16.2,1.6,47.246
2,1,3,2024-07-07 14:06:21.461000+00:00,0.0,62.0,16.3,2.0,18.957
3,1,4,2024-07-07 14:07:53.675000+00:00,0.0,61.0,16.3,1.9,51.154
4,1,5,2024-07-07 14:09:25.527000+00:00,0.0,60.0,16.2,2.2,22.990
5,1,6,2024-07-07 14:10:57.353000+00:00,0.0,60.0,16.3,2.4,54.814
6,1,7,2024-07-07 14:12:29.113000+00:00,0.0,61.0,16.3,2.1,26.554
7,1,8,2024-07-07 14:14:01.043000+00:00,0.0,60.0,16.4,2.4,58.476
8,1,9,2024-07-07 14:15:33.002000+00:00,0.0,61.0,16.5,2.0,30.422
9,1,10,2024-07-07 14:17:04.948000+00:00,0.0,61.0,16.8,1.5,2.385


In [75]:
model_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect",
    "track_evolution",
    "track_evolution_available",
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed"
]

target = "lap_duration"

pace_model_data_weather = enhanced_lap_data_weather[
    model_features + [target, "lap_number"]
].copy()

print("Shape:", pace_model_data_weather.shape)

print("\nMissing values:")
display(
    pace_model_data_weather.isna().sum()
)

print("\nColumns:")
print(pace_model_data_weather.columns.tolist())

KeyError: "['track_evolution_available'] not in index"

In [76]:
# Inspect current enhanced dataset columns

print("enhanced_lap_data_weather columns:")
for i, col in enumerate(enhanced_lap_data_weather.columns):
    print(i, col)

print("\nExisting pace_model_data columns:")
for i, col in enumerate(pace_model_data.columns):
    print(i, col)

enhanced_lap_data_weather columns:
0 meeting_key
1 session_key
2 driver_number
3 lap_number
4 date_start
5 duration_sector_1
6 duration_sector_2
7 duration_sector_3
8 i1_speed
9 i2_speed
10 is_pit_out_lap
11 lap_duration
12 segments_sector_1
13 segments_sector_2
14 segments_sector_3
15 st_speed
16 stint_number
17 compound
18 tyre_age
19 fuel_effect
20 position
21 traffic_ahead
22 traffic_behind
23 position_change
24 tyre_degradation_effect
25 track_evolution
26 rainfall
27 humidity
28 air_temperature
29 wind_speed
30 weather_lag_seconds

Existing pace_model_data columns:
0 meeting_key
1 session_key
2 driver_number
3 lap_number
4 date_start
5 duration_sector_1
6 duration_sector_2
7 duration_sector_3
8 i1_speed
9 i2_speed
10 is_pit_out_lap
11 lap_duration
12 segments_sector_1
13 segments_sector_2
14 segments_sector_3
15 st_speed
16 stint_number
17 compound
18 tyre_age
19 fuel_effect
20 position
21 traffic_ahead
22 traffic_behind
23 position_change
24 tyre_degradation_effect
25 track_evol

In [77]:
# Add weather features to the existing pace_model_data

weather_for_model = enhanced_lap_data_weather[
    [
        "driver_number",
        "lap_number",
        "rainfall",
        "humidity",
        "air_temperature",
        "wind_speed",
        "weather_lag_seconds"
    ]
].copy()

pace_model_data_weather = pace_model_data.merge(
    weather_for_model,
    on=["driver_number", "lap_number"],
    how="left",
    validate="one_to_one"
)

print("Original pace_model_data shape:", pace_model_data.shape)
print("Weather-enhanced shape:", pace_model_data_weather.shape)

print("\nWeather missing values:")
display(
    pace_model_data_weather[
        [
            "rainfall",
            "humidity",
            "air_temperature",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].isna().sum()
)

print("\nTrack evolution availability:")
display(
    pace_model_data_weather["track_evolution_available"]
    .value_counts(dropna=False)
)

Original pace_model_data shape: (668, 27)
Weather-enhanced shape: (668, 32)

Weather missing values:


rainfall               0
humidity               0
air_temperature        0
wind_speed             0
weather_lag_seconds    0
dtype: int64


Track evolution availability:


track_evolution_available
1    574
0     94
Name: count, dtype: int64

In [78]:
# Inspect final weather-enhanced modeling dataset

print("Final modeling columns:\n")

for i, col in enumerate(pace_model_data_weather.columns):
    print(i, col)

print("\nTarget statistics:")
display(
    pace_model_data_weather["lap_duration"].describe()
)

print("\nLap number range:")
print(
    "Min:", pace_model_data_weather["lap_number"].min()
)
print(
    "Max:", pace_model_data_weather["lap_number"].max()
)

print("\nRows by chronological split:")

train_mask = pace_model_data_weather["lap_number"] < 45
test_mask = pace_model_data_weather["lap_number"] >= 45

print("Train rows:", train_mask.sum())
print("Test rows:", test_mask.sum())

print("\nTrain lap range:")
print(
    pace_model_data_weather.loc[train_mask, "lap_number"].min(),
    "to",
    pace_model_data_weather.loc[train_mask, "lap_number"].max()
)

print("\nTest lap range:")
print(
    pace_model_data_weather.loc[test_mask, "lap_number"].min(),
    "to",
    pace_model_data_weather.loc[test_mask, "lap_number"].max()
)

Final modeling columns:

0 meeting_key
1 session_key
2 driver_number
3 lap_number
4 date_start
5 duration_sector_1
6 duration_sector_2
7 duration_sector_3
8 i1_speed
9 i2_speed
10 is_pit_out_lap
11 lap_duration
12 segments_sector_1
13 segments_sector_2
14 segments_sector_3
15 st_speed
16 stint_number
17 compound
18 tyre_age
19 fuel_effect
20 position
21 traffic_ahead
22 traffic_behind
23 position_change
24 tyre_degradation_effect
25 track_evolution
26 track_evolution_available
27 rainfall
28 humidity
29 air_temperature
30 wind_speed
31 weather_lag_seconds

Target statistics:


count    668.000000
mean      93.058708
std        3.567319
min       88.293000
25%       91.140000
50%       92.444500
75%       93.480250
max      118.710000
Name: lap_duration, dtype: float64


Lap number range:
Min: 1
Max: 52

Rows by chronological split:
Train rows: 535
Test rows: 133

Train lap range:
1 to 44

Test lap range:
45 to 52
